<a href="https://colab.research.google.com/github/fortidatasolutions-ctrl/train/blob/main/DanMetadata_Writer_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2024 The AI Edge Authors.

### Creating and Activating a Virtual Environment in Colab (Demonstration)

While Colab provides an isolated environment for each session, you can still create and use a virtual environment within it if you need to. Note that the virtual environment will be created in the current working directory (`/content/`).

In [ ]:
# Remove existing virtual environment if it exists to ensure a clean setup
!rm -rf .venv

# Create a virtual environment named '.venv'
!python -m venv .venv

print("Ensuring pip and setuptools are installed and upgraded in the virtual environment...")
import os
import sys

venv_python_executable = "./.venv/bin/python"
venv_pip_executable = "./.venv/bin/pip"

# Check if pip executable exists. If not, install it via get-pip.py
# This handles cases where python -m venv fails to install pip properly due to `ensurepip` issues.
if not os.path.exists(venv_pip_executable):
    print("pip executable not found in .venv. Attempting manual installation with get-pip.py...")
    # Download get-pip.py into the content directory
    !curl -sSL https://bootstrap.pypa.io/get-pip.py -o get-pip.py
    # Install pip into the virtual environment using its python interpreter
    !{venv_python_executable} get-pip.py
    # Clean up get-pip.py
    !rm get-pip.py
    print("pip installed in .venv using get-pip.py.")
else:
    print("pip executable found in .venv.")

# Now that pip should definitely be present, ensure it's upgraded along with setuptools
print("Upgrading pip and setuptools in the virtual environment...")
!{venv_python_executable} -m pip install --upgrade pip setuptools

print("Virtual environment created and pip/setuptools ensured: .venv")

# Add the virtual environment's site-packages to sys.path immediately after creation
venv_site_packages = os.path.join("/content/.venv", "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")

if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)
    print(f"Added {venv_site_packages} to sys.path.")
else:
    print(f"{venv_site_packages} is already in sys.path.")


In [ ]:
# To use the virtual environment's Python interpreter and install packages:
# We will use the full path to the python executable within the venv.
!/content/.venv/bin/pip install pandas

print("Pandas installed in the virtual environment.")

# Verify that pandas is available in this specific environment
!/content/.venv/bin/python -c "import pandas; print(pandas.__version__)"

In [ ]:
# Ensure TensorFlow and Keras are imported.
# We are no longer explicitly installing a specific TensorFlow version here
# as TFODAPI typically works with the pre-installed Colab TensorFlow, and we moved away from
# tensorflow-lite-model-maker's specific TF 2.10.0 requirement.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

Please note that the activation of a virtual environment using `source` commands in `!shell` commands within Colab cells usually only applies to that specific cell's subshell. For persistent use across multiple cells, you would typically refer to the full path of the Python executable and `pip` within the `.venv` (e.g., `!/content/.venv/bin/python` or `!/content/.venv/bin/pip`).

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow Lite Metadata Writer API



<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/lite/models/convert/metadata_writer_tutorial"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/tensorflow/blob/master/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/tensorflow/blob/master/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/tensorflow/tensorflow/lite/g3doc/models/convert/metadata_writer_tutorial.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>

</table>

[TensorFlow Lite Model Metadata](https://www.tensorflow.org/lite/models/convert/metadata) is a standard model description format. It contains rich semantics for general model information, inputs/outputs, and associated files, which makes the model self-descriptive and exchangeable.

Model Metadata is currently used in the following two primary use cases:
1. **Enable easy model inference using TensorFlow Lite [Task Library](https://www.tensorflow.org/lite/inference_with_metadata/task_library/overview) and [codegen tools](https://www.tensorflow.org/lite/inference_with_metadata/codegen)**. Model Metadata contains the mandatory information required during inference, such as label files in image classification, sampling rate of the audio input in audio classification, and tokenizer type to process input string in Natural Language models.

2. **Enable model creators to include documentation**, such as the description of model inputs/outputs or how to use the model. Model users can view this documentation via visualization tools such as [Netron](https://netron.app/).

TensorFlow Lite Metadata Writer API provides an easy-to-use API to create Model Metadata for popular ML tasks supported by the TFLite Task Library.  This notebook shows examples of how the metadata should be populated for the following tasks below:

*   [Image classifiers](#image_classifiers)
*   [Object detectors](#object_detectors)
*   [Image segmenters](#image_segmenters)
*   [Natural language classifiers](#nl_classifiers)
*   [Audio classifiers](#audio_classifiers)

Metadata writers for BERT natural language classifiers and BERT question answerers are coming soon.

If you want to add metadata for use cases that are not supported, please use the [Flatbuffers Python API](https://www.tensorflow.org/lite/models/convert/metadata#adding_metadata). See the tutorials [here](https://www.tensorflow.org/lite/models/convert/metadata#adding_metadata).


## Prerequisites

Install the TensorFlow Lite Support Pypi package.

In [ ]:
# Install mediapipe into the virtual environment
!/content/.venv/bin/pip install mediapipe

In [ ]:
# Install tflite-support into the virtual environment
!/content/.venv/bin/pip install tflite-support

In [ ]:
# Add the virtual environment's site-packages to sys.path
import sys
import os

venv_site_packages = os.path.join("/content/.venv", "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")

if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)
    print(f"Added {venv_site_packages} to sys.path.")
else:
    print(f"{venv_site_packages} is already in sys.path.")

# Verify that tflite_support can now be imported from the venv
try:
    import tflite_support
    print(f"Successfully imported tflite_support from the virtual environment.")
except ImportError:
    print("Failed to import tflite_support. Check your virtual environment setup.")

### Using MediaPipe Tasks

MediaPipe Tasks provides a suite of pre-trained models and tools for common machine learning tasks, especially for on-device applications. Here's a basic example of how to use MediaPipe's Image Classifier.

In [ ]:
# The line below is commented out to prevent protobuf version conflicts with TensorFlow.
# !pip install --force-reinstall protobuf

# First, let's download a sample image to classify.
!wget -q -O image.jpg https://storage.googleapis.com/mediapipe-assets/cat.jpg

# Import MediaPipe modules
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load the pre-trained Image Classifier model (e.g., efficientnet_lite0)
model_path = 'efficientnet_lite0.tflite'
!wget -q -O {model_path} https://storage.googleapis.com/mediapipe-models/image_classifier/efficientnet_lite0/float32/1/efficientnet_lite0.tflite

# Configure the Image Classifier options
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.ImageClassifierOptions(base_options=base_options, score_threshold=0.25)
classifier = vision.ImageClassifier.create_from_options(options)

# Load the input image
image = mp.Image.create_from_file('image.jpg')

# Classify the image
classification_result = classifier.classify(image)

# Display the results
for category in classification_result.classifications[0].categories:
    print(f"Category: {category.category_name}, Score: {category.score:.2f}")

In [ ]:
# Upgrade Pillow to resolve compatibility issues with apache-beam during TFODAPI installation.
# apache-beam requires pillow<13,>=12.1.1
!/content/.venv/bin/pip install --upgrade pillow==12.1.1

print("Pillow upgraded to 12.1.1.")

In this example:

1.  We downloaded a sample image (`cat.jpg`) and a pre-trained `efficientnet_lite0.tflite` image classification model.
2.  We imported the necessary `mediapipe.tasks.python` and `mediapipe.tasks.python.vision` modules.
3.  An `ImageClassifier` was created using `create_from_options`, specifying the model path and a score threshold.
4.  The image was loaded using `mp.Image.create_from_file`.
5.  The `classify` method was called on the `classifier` object to get the prediction results.
6.  Finally, the detected categories and their confidence scores were printed.

## Model Training: Fine-tuning MobileNetV2 and Training a New Model

This section will cover the steps to fine-tune an existing `mobilenet_v2` model using transfer learning and also outline the process for training a new model from scratch. We'll start with setting up the TensorFlow environment.

In [ ]:
# Define global image dimensions and number of classes for the model
# IMG_SIZE = (224, 224)
# IMG_SHAPE = IMG_SIZE + (3,)
# NUM_CLASSES = 10 # For CIFAR-10 dataset


### Step 1: Prepare the Dataset

Before we can train or fine-tune a model, we need a dataset. For this example, we'll assume a dataset is available for image classification. You would typically load, preprocess, and split your data into training, validation, and test sets here.

In [ ]:
import sys
import os
# Add the virtual environment's site-packages to sys.path
venv_site_packages = os.path.join("/content/.venv", "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

import tensorflow as tf
from tensorflow import keras

# Define global image dimensions and number of classes for the model
IMG_SIZE = (224, 224)
IMG_SHAPE = IMG_SIZE + (3,)
NUM_CLASSES = 10 # For CIFAR-10 dataset

# Placeholder for data loading and preprocessing
# For demonstration, we'll create a dummy dataset or use a standard one like CIFAR-10 or ImageNet subset.

# Example: Load a small dataset for demonstration (e.g., CIFAR-10)
# (Uncomment and run if you want to use CIFAR-10)
# (Note: This will download the dataset if not already present)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(f"Raw Training data shape: {x_train.shape}, {y_train.shape}")
print(f"Raw Test data shape: {x_test.shape}, {y_test.shape}")

# Preprocess the data for MobileNetV2
def preprocess_data(images, labels):
    images = tf.image.resize(images, IMG_SIZE) # Resize images to 224x224
    images = keras.applications.mobilenet_v2.preprocess_input(images) # Scale pixels to [-1, 1]
    labels = tf.squeeze(labels) # Remove single-dimensional entries from the shape of an array
    return images, labels

# Create TensorFlow datasets
# Use the number of training samples for shuffle buffer size
SHUFFLE_BUFFER_SIZE = x_train.shape[0]
BATCH_SIZE = 1 # Further reduced batch size to mitigate OOM errors
PREFETCH_BUFFER_SIZE = tf.data.AUTOTUNE # Use tf.data.AUTOTUNE for optimal buffer size

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=PREFETCH_BUFFER_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
val_ds = val_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=PREFETCH_BUFFER_SIZE)

print(f"Processed Training dataset element spec: {train_ds.element_spec}")
print(f"Processed Validation dataset element spec: {val_ds.element_spec}")

# In a real scenario, you would use `tf.keras.utils.image_dataset_from_directory` or `tf.data.Dataset`
# for more robust and scalable data pipelines.


---

### Step 2: Fine-tuning MobileNetV2

We'll use a pre-trained MobileNetV2 model as a base and add a custom classification head on top. This is a common approach in transfer learning.

In [ ]:
import sys
import os
# Add the virtual environment's site-packages to sys.path
venv_site_packages = os.path.join("/content/.venv", "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.backend.clear_session()

# Create the base model from the pre-trained MobileNetV2 model
base_model = keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                           include_top=False, # Don't include the top classification layer
                                           weights='imagenet')

# Freeze the base model to prevent its weights from being updated during the first training phase
base_model.trainable = False

# Create a new model on top
inputs = keras.Input(shape=IMG_SHAPE)
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


### Step 2.1: Train the Fine-tuned Model

Now, we'll train the classification head of our fine-tuned `MobileNetV2` model using the prepared dataset. We'll specify the number of epochs and use the validation dataset to monitor performance.

In [ ]:
#history = model.fit(train_ds, epochs=3, validation_data=val_ds)

print("Model training complete!")

This sets up the `mobilenet_v2` model for fine-tuning. The `base_model` is frozen initially, and a new classification head is added. The next step would be to train this new head on your specific dataset. After that, you can unfreeze some layers of the `base_model` and fine-tune the entire model with a lower learning rate.

## Implementing the Swedish Driving AI Training Pipeline for MobileNetV2

Based on your detailed document, we will now shift our focus to training an object detection model, specifically for Swedish traffic signs, using `SSD MobileNetV2` or `EfficientDet Lite` with TensorFlow Lite Model Maker.

### Install TensorFlow Lite Model Maker

First, we need to install the `tensorflow-lite-model-maker` library. Since we are using a virtual environment, we will install it there.

In [ ]:
# Install tensorflow-lite-model-maker into the current Colab environment
!pip install tensorflow-lite-model-maker

print("TensorFlow Lite Model Maker installed.")

### Switching to TensorFlow Object Detection API

Given the persistent compatibility issues with `tensorflow-lite-model-maker` and our current Colab environment (Python 3.12, TensorFlow 2.21.0), we will proceed with the **TensorFlow Object Detection API (TFODAPI)** for training your Swedish traffic sign object detection model. This API is robust and compatible with our current setup.

Below are the steps to install and configure the TFODAPI. Please execute these cells sequentially.

#### 1. Install Dependencies for TFODAPI

We need to install several libraries required by the TensorFlow Object Detection API. This includes `lxml`, `matplotlib`, and `pycocotools`.

In [ ]:
# Install necessary dependencies for TFODAPI into the virtual environment
!/content/.venv/bin/pip install lxml matplotlib pycocotools tensorflow-io

print("TFODAPI dependencies installed.")

#### 2. Clone the TensorFlow Models Repository

The TensorFlow Object Detection API is part of the `tensorflow/models` repository. We need to clone this repository to access the necessary scripts and configurations.

In [ ]:
# Clone the tensorflow/models repository if not already present
import os
if not os.path.exists('models'):
    !git clone https://github.com/tensorflow/models.git
else:
    print("TensorFlow models repository already cloned.")

print("Cloned TensorFlow models repository.")

#### 3. Compile Protobuf Files

The TensorFlow Object Detection API uses Protocol Buffers (`.proto` files) for model configurations. We need to compile these into Python-readable files. This is a crucial step for TFODAPI setup.

# This cell is now used to install a compatible protobuf version for TensorFlow and TFODAPI into the virtual environment.
# Installing protobuf==4.21.6 which is known to be compatible with TensorFlow 2.21.0.
# This should resolve the ImportError encountered during TFODAPI verification.
!/content/.venv/bin/pip install protobuf==4.21.6

print("Protobuf 4.21.6 installed. Please restart your Colab runtime NOW to apply changes and then re-execute the TFODAPI setup cells.")

In [ ]:
print('Running c28c1259')
# This cell is now used to install a compatible protobuf version for TensorFlow and TFODAPI into the virtual environment.
# Installing protobuf==4.21.6 which is known to be compatible with TensorFlow 2.21.0.
# This should resolve the ImportError encountered during TFODAPI verification.
!/content/.venv/bin/pip install protobuf==4.21.6

print("Protobuf 4.21.6 installed. Please restart your Colab runtime NOW to apply changes and then re-execute the TFODAPI setup cells.")

In [ ]:
# Navigate to the research directory
%cd /content/models/research

# Compile protobuf files
# The --python_out argument should point to the current directory
!protoc object_detection/protos/*.proto --python_out=.

# Check if any .py files were generated to confirm compilation
!ls object_detection/protos/*.py

print("Protobuf files compiled.")

#### 4. Install the Object Detection API as a Python Package

Now we need to install the `object_detection` directory as a Python package. This makes its modules importable.

In [ ]:
# Add models/research and models/research/slim to PYTHONPATH permanently for this session
import os
import sys
research_path = '/content/models/research'
slim_path = '/content/models/research/slim'
if research_path not in sys.path:
    sys.path.append(research_path)
if slim_path not in sys.path:
    sys.path.append(slim_path)

# Navigate to the research directory to install object_detection
%cd {research_path}

# List contents of the current directory for debugging
print(f"Contents of {os.getcwd()}:")
!ls -F

# Install the object_detection API using setup.py install from the correct subdirectory
# For TensorFlow 2.x, setup.py is typically in object_detection/packages/tf2/
!/content/.venv/bin/python object_detection/packages/tf2/setup.py install

# Go back to content directory
%cd /content

print("TensorFlow Object Detection API installed.")

In [ ]:
import os

# Define the path to the file that needs patching
file_to_patch = '/content/.venv/lib/python3.12/site-packages/object_detection/core/freezable_sync_batch_norm.py'

print(f"Attempting to patch: {file_to_patch}")

# Read the content of the file
with open(file_to_patch, 'r') as f:
    content = f.read()

# Replace the problematic line
# Now looking for 'tf.keras.layers.SyncBatchNormalization'
# Replacing with 'tf.keras.layers.BatchNormalization' as a workaround
old_line_part = 'tf.keras.layers.SyncBatchNormalization'
new_line_part = 'tf.keras.layers.BatchNormalization' # CRITICAL CHANGE

if old_line_part in content:
    new_content = content.replace(old_line_part, new_line_part)

    # Write the modified content back to the file
    with open(file_to_patch, 'w') as f:
        f.write(new_content)
    print("Successfully patched 'freezable_sync_batch_norm.py' by replacing SyncBatchNormalization with BatchNormalization.")
else:
    print(f"'{old_line_part}' not found in the file. Patching might not be necessary, the file has already been patched, or the content is unexpected.")

print("Please re-run the TFODAPI verification cell (`f3394e7e`) to confirm the fix.")

In [ ]:
# Install tf-models-official, which contains the 'official' module required by TFODAPI
!/content/.venv/bin/pip install tf-models-official

print("tf-models-official installed.")
print("Please re-run the TFODAPI verification cell (f3394e7e) after this installation to confirm the fix.")

In [ ]:
import os

# Define the path to the file that needs patching
file_to_patch = '/content/.venv/lib/python3.12/site-packages/object_detection/inputs.py'

print(f"Attempting to patch: {file_to_patch}")

# Read the content of the file
with open(file_to_patch, 'r') as f:
    content = f.read()

# The problematic line is 'from tensorflow.compat.v1 import estimator as tf_estimator'
# We need to change it to 'import tensorflow_estimator as tf_estimator'
old_line = 'from tensorflow.compat.v1 import estimator as tf_estimator'
new_line = 'import tensorflow_estimator as tf_estimator'

if old_line in content:
    new_content = content.replace(old_line, new_line)

    # Write the modified content back to the file
    with open(file_to_patch, 'w') as f:
        f.write(new_content)
    print("Successfully patched 'inputs.py' to directly import tensorflow_estimator.")
else:
    print(f"'{old_line}' not found in the file. Patching might not be necessary, the file has already been patched, or the content is unexpected.")

print("Please re-run the TFODAPI verification cell (`f3394e7e`) and then the training cell (`3ea6e3e4`).")

#### 5. Verify Installation (Optional but Recommended)

Let's run a quick test to ensure the Object Detection API is correctly installed and accessible.

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import tensorflow_io as tfio # Explicitly import tensorflow_io
import object_detection
import tensorflow as tf

print("TFODAPI and TensorFlow imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Check if object_detection can load a model builder
from object_detection.builders import model_builder
print("Model builder import successful.")

print("TFODAPI setup complete and verified!")

Now that the TensorFlow Object Detection API is set up, the next critical step is to ensure your dataset is prepared and available. Please make sure your Swedish traffic sign dataset (images and Pascal VOC annotations) is uploaded and unzipped into the `/content/data/images` and `/content/data/annotations` directories, as instructed in cell `1e1012ed`.

In [ ]:
# Create the 'images' and 'annotations' directories if they don't exist
import os

base_data_dir = '/content/data'
images_dir = os.path.join(base_data_dir, 'images')
annotations_dir = os.path.join(base_data_dir, 'annotations')

os.makedirs(images_dir, exist_ok=True)
os.makedirs(annotations_dir, exist_ok=True)

print(f"Created directories: {images_dir} and {annotations_dir}")

# Move image files (.jpg, .png) to the 'images' directory
# Move annotation files (.xml) to the 'annotations' directory

files_in_data_dir = [f for f in os.listdir(base_data_dir) if os.path.isfile(os.path.join(base_data_dir, f))]

for filename in files_in_data_dir:
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        os.rename(os.path.join(base_data_dir, filename), os.path.join(images_dir, filename))
    elif filename.lower().endswith(('.xml')):
        os.rename(os.path.join(base_data_dir, filename), os.path.join(annotations_dir, filename))

print("Files moved to their respective directories.")

In [ ]:
# Re-run the data verification code to confirm organization
import os

images_dir = '/content/data/images'
anotations_dir = '/content/data/annotations'

print(f"Verifying directory: {images_dir}")
if not os.path.exists(images_dir) or not os.listdir(images_dir):
    print(f"Warning: Images directory not found or empty at {images_dir}. Please ensure your dataset is correctly uploaded and unzipped.")
else:
    print(f"Images directory found and contains {len(os.listdir(images_dir))} files.")

print(f"Verifying directory: {annotations_dir}")
if not os.path.exists(annotations_dir) or not os.listdir(annotations_dir):
    print(f"Warning: Annotations directory not found or empty at {annotations_dir}. Please ensure your dataset is correctly uploaded and unzipped.")
else:
    print(f"Annotations directory found and contains {len(os.listdir(annotations_dir))} files.")

print(f"Configured image directory: {images_dir}")
print(f"Configured annotations directory: {annotations_dir}")

### 3. Complete AI Training Process (as per your document)

Let's follow the steps outlined in your document to prepare the dataset and train the model.

#### STEP 1 — Collect Datasets: Swedish Traffic Sign Images

Your document highlights the need for a comprehensive dataset of Swedish Traffic Sign Images, recommending datasets like the Swedish Traffic Signs Dataset or Mapillary Traffic Sign Dataset.

**Action Required**: You will need to upload or link your prepared dataset (e.g., in Pascal VOC format) to this Colab environment. Ensure your dataset includes images for all the recommended sign classes (e.g., `speed_limit_30`, `stop_sign`, `pedestrian_crossing`, etc.).

For example, you might upload a zip file containing `images` and `annotations` directories and then unzip it here.

In [ ]:
# --- Data Loading and Preparation (Action Required) ---
# To train the object detection model, you need a dataset of Swedish traffic signs.
# This dataset should be prepared in a format like Pascal VOC, with:
#   - An 'images' directory containing your image files.
#   - An 'annotations' directory containing XML annotation files for each image.

# Potential Data Sources (as provided by you):
# - https://commons.wikimedia.org/wiki/Road_signs_in_Sweden: Templates for reference.
# - https://trafficdata.se/dataset/: A potential source for traffic data, which you may need to process and annotate.

# --- Instructions ---
# 1. Acquire or create your Swedish traffic sign dataset in Pascal VOC format.
# 2. Package your 'images' and 'annotations' directories into a single zip file (e.g., 'swedish_traffic_signs.zip').
# 3. Upload this zip file to your Colab environment (e.g., by dragging it into the files panel on the left).
# 4. Uncomment and run the `!unzip` command below, ensuring the filename and destination path are correct.

# Example: If your zip file is named 'swedish_traffic_signs.zip' and you want to extract it to '/content/data/'
# If you have not uploaded your dataset yet, this cell will run but will not find any data.

# !unzip /content/swedish_traffic_signs.zip -d /content/data/

# Define the paths to your dataset directories
# These paths should point to where your images and annotation files are located after unzipping.
images_dir = '/content/data/images'
annotations_dir = '/content/data/annotations'

# Verify if the directories exist (optional, but good for debugging)
import os
if not os.path.exists(images_dir):
    print(f"Warning: Images directory not found at {images_dir}. Please ensure your dataset is correctly uploaded and unzipped.")
if not os.path.exists(annotations_dir):
    print(f"Warning: Annotations directory not found at {annotations_dir}. Please ensure your dataset is correctly uploaded and unzipped.")

print(f"Configured image directory: {images_dir}")
print(f"Configured annotations directory: {annotations_dir}")


#### STEP 2 — Clean and Organize Data & STEP 3 — Label / Annotate Data

Your document emphasizes the importance of cleaning, organizing, and annotating data. It suggests tools like LabelImg, CVAT, or Roboflow for bounding box annotations around traffic signs.

**Note**: This step is typically done offline before bringing the data into Colab. We assume your dataset is already prepared and annotated in a format like Pascal VOC, which `tensorflow-lite-model-maker` can directly use.

#### STEP 4 — Create Classes: Recommended Swedish Traffic Sign Classes

Based on your document, here are the recommended traffic sign classes. You will need to create a `label_map` for these classes, which maps each class name to an integer ID.

In [ ]:
# Define your label map based on the classes from your document
# This needs to correspond to the labels in your annotation files.
label_map = {
    '1': 50,
    '4': 67,
    '5': 56,
    '21': 26,
    '23': 27,
    '26': 28,
    '27': 1,
    '33': 51,
    '36': 61,
    '38': 29,
    '46': 60,
    '48': 55,
    '55': 30,
    '56': 70,
    '57': 57,
    '58': 72,
    '59': 31,
    '60': 32,
    '62': 2,
    '64': 3,
    '65': 52,
    '66': 4,
    '69': 5,
    '70': 33,
    '73': 6,
    '82': 7,
    '83': 34,
    '89': 8,
    '90': 66,
    '91': 35,
    '93': 9,
    '95': 10,
    '99': 64,
    '102': 11,
    '103': 36,
    '108': 59,
    '111': 37,
    '112': 12,
    '113': 49,
    '115': 58,
    '116': 38,
    '117': 13,
    '118': 14,
    '124': 39,
    '125': 15,
    '128': 69,
    '129': 40,
    '131': 41,
    '135': 42,
    '137': 43,
    '139': 16,
    '144': 44,
    '147': 63,
    '149': 17,
    '150': 45,
    '152': 54,
    '156': 65,
    '159': 68,
    '165': 46,
    '166': 47,
    '169': 18,
    '170': 19,
    '171': 20,
    '172': 71,
    '173': 62,
    '174': 21,
    '177': 53,
    '178': 22,
    '179': 48,
    '180': 23,
    '181': 24,
    'Varning för farlig kurva': 25
}

# Reverse map for convenience (optional)
class_names = [name for name, id in sorted(label_map.items(), key=lambda item: item[1])]
print("Defined traffic sign classes and label map.")

### Preparing Data for TensorFlow Object Detection API

To train with the TensorFlow Object Detection API, your dataset needs to be in `TFRecord` format, and you need a `label_map.pbtxt` file. We will now generate these files and prepare the pipeline configuration.

In [ ]:
# Create label_map.pbtxt for TFODAPI
import os

output_dir = '/content/data/tfodapi_data'
os.makedirs(output_dir, exist_ok=True)

label_map_path = os.path.join(output_dir, 'label_map.pbtxt')

with open(label_map_path, 'w') as f:
    for name, id in label_map.items():
        f.write('item {\n')
        f.write(f'  id: {id}\n')
        f.write(f"  name: '{name}'\n")
        f.write('}\n')

print(f"label_map.pbtxt created at: {label_map_path}")

Next, we need to convert your Pascal VOC XML annotations and images into TFRecord format. This script will parse the XML files, create a CSV of bounding box information, and then generate `train.record` and `val.record` files.

In [ ]:
import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import io
import tensorflow as tf
from PIL import Image

# Ensure the necessary TFODAPI utilities are in sys.path
# These were added during the TFODAPI installation step, but re-adding for clarity/safety.
research_path = '/content/models/research'
slim_path = '/content/models/research/slim'
if research_path not in sys.path:
    sys.path.append(research_path)
if slim_path not in sys.path:
    sys.path.append(slim_path)

# Import TFODAPI specific utilities
from object_detection.utils import dataset_util

# Define paths (using the global variables already in kernel state)
# images_dir = '/content/data/images'
# annotations_dir = '/content/data/annotations'
# output_dir = '/content/data/tfodapi_data' (already defined above)

# Function to parse XML and convert to DataFrame
def xml_to_csv(path, class_names_list):
    xml_list = []
    for xml_file in glob.glob(os.path.join(path, '*.xml')):
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Get image width and height from the <size> element
        img_width = int(root.find('size').find('width').text) if root.find('size') is not None and root.find('size').find('width') is not None else 0
        img_height = int(root.find('size').find('height').text) if root.find('size') is not None and root.find('size').find('height') is not None else 0

        for member in root.findall('object'):
            class_name = member.find('name').text

            # Ensure the class name is in our defined label_map
            if class_name not in class_names_list:
                print(f"Warning: Class '{class_name}' found in {xml_file} but not in label_map. Skipping this object.")
                continue

            bndbox = member.find('bndbox')
            if bndbox is None: # Skip if no bounding box found for the object
                print(f"Warning: Bounding box not found for object '{class_name}' in {xml_file}. Skipping this object.")
                continue

            value = (
                root.find('filename').text,
                img_width,
                img_height,
                class_name,
                int(bndbox.find('xmin').text),
                int(bndbox.find('ymin').text),
                int(bndbox.find('xmax').text),
                int(bndbox.find('ymax').text)
            )
            xml_list.append(value)
    column_name = ['filename', 'width', 'height', 'class', 'xmin', 'ymin', 'xmax', 'ymax']
    xml_df = pd.DataFrame(xml_list, columns=column_name)
    return xml_df

# Function to create TFRecord from DataFrame
def create_tf_example(group, path, label_map_dict):
    with tf.io.gfile.GFile(os.path.join(path, f"{group.filename.iloc[0]}"), 'rb') as fid:
        encoded_jpg = fid.read()
    encoded_jpg_io = io.BytesIO(encoded_jpg)
    image = Image.open(encoded_jpg_io)
    width, height = image.size

    filename = group.filename.iloc[0].encode('utf8')
    image_format = b'jpg'
    xmins = []
    xmaxs = []
    ymins = []
    ymaxs = []
    classes_text = []
    classes = []

    for index, row in group.iterrows():
        xmins.append(row['xmin'] / width)
        xmaxs.append(row['xmax'] / width)
        ymins.append(row['ymin'] / height)
        ymaxs.append(row['ymax'] / height)
        classes_text.append(row['class'].encode('utf8'))
        classes.append(label_map_dict[row['class']])

    tf_example = tf.train.Example(features=tf.train.Features(feature={
        'image/height': dataset_util.int64_feature(height),
        'image/width': dataset_util.int64_feature(width),
        'image/filename': dataset_util.bytes_feature(filename),
        'image/source_id': dataset_util.bytes_feature(filename),
        'image/encoded': dataset_util.bytes_feature(encoded_jpg),
        'image/format': dataset_util.bytes_feature(image_format),
        'image/object/bbox/xmin': dataset_util.float_list_feature(xmins),
        'image/object/bbox/xmax': dataset_util.float_list_feature(xmaxs),
        'image/object/bbox/ymin': dataset_util.float_list_feature(ymins),
        'image/object/bbox/ymax': dataset_util.float_list_feature(ymaxs),
        'image/object/class/text': dataset_util.bytes_list_feature(classes_text),
        'image/object/class/label': dataset_util.int64_list_feature(classes),
    }))
    return tf_example

# Get a list of class names from the label_map for validation
class_names_list = list(label_map.keys())

# Convert all XMLs to a single DataFrame
full_labels_df = xml_to_csv(annotations_dir, class_names_list)

# Handle case where full_labels_df might be empty after filtering
if full_labels_df.empty:
    print("Error: No valid annotations found after parsing XML files and filtering by label_map. Cannot create TFRecords.")
else:
    # Get unique filenames for splitting and convert to list to ensure compatibility with sklearn.model_selection.train_test_split
    unique_filenames = full_labels_df['filename'].unique().tolist()

    # Split filenames into training and validation sets
    # Ensure there are enough samples to split
    if len(unique_filenames) < 2:
        print(f"Warning: Only {len(unique_filenames)} unique images found. Cannot perform train-test split. Using all data for training.")
        train_filenames = unique_filenames
        val_filenames = [] # No validation set
    else:
        train_filenames, val_filenames = train_test_split(
            unique_filenames, test_size=0.2, random_state=42
        )

    # Filter the full DataFrame to create train and validation DataFrames
    train_df = full_labels_df[full_labels_df['filename'].isin(train_filenames)]
    val_df = full_labels_df[full_labels_df['filename'].isin(val_filenames)]

    # Group by filename for TFRecord creation
    grouped_train = train_df.groupby('filename')
    grouped_val = val_df.groupby('filename')

    # Create TFRecords
    train_record_path = os.path.join(output_dir, 'train.record')
    val_record_path = os.path.join(output_dir, 'val.record')

    print(f"Creating TFRecord for training data at {train_record_path}...")
    with tf.io.TFRecordWriter(train_record_path) as writer:
        for filename, group in grouped_train:
            tf_example = create_tf_example(group, images_dir, label_map)
            writer.write(tf_example.SerializeToString())
    print("Training TFRecord created.")

    print(f"Creating TFRecord for validation data at {val_record_path}...")
    with tf.io.TFRecordWriter(val_record_path) as writer:
        for filename, group in grouped_val:
            tf_example = create_tf_example(group, images_dir, label_map)
            writer.write(tf_example.SerializeToString())
    print("Validation TFRecord created.")

    print(f"Generated {len(grouped_train)} training examples and {len(grouped_val)} validation examples.")

### Identify Unique Class Names in Annotations

The previous step failed because the class names in your XML annotation files do not match the `label_map` defined. Use the following cell to extract all unique class names directly from your XML files. You will then need to update the `label_map` in cell `3fc8dd0a` with these correct class names and their corresponding IDs.

In [ ]:
import os
import glob
import xml.etree.ElementTree as ET

# Assuming annotations_dir is already defined and points to your XML files
# annotations_dir = '/content/data/annotations'

if not os.path.exists(annotations_dir):
    print(f"Error: Annotations directory not found at {annotations_dir}.")
else:
    unique_classes = set()
    for xml_file in glob.glob(os.path.join(annotations_dir, '*.xml')):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        for member in root.findall('object'):
            unique_classes.add(member[0].text)

    if unique_classes:
        print("Unique class names found in your annotation XML files:")
        for cls_name in sorted(list(unique_classes)):
            print(f"- {cls_name}")
        print("\nPlease update your `label_map` in cell `3fc8dd0a` with these names and then re-run the TFRecord generation cells.")
    else:
        print("No object classes found in the XML annotation files. Please ensure your XMLs are correctly formatted and contain objects.")


Now that your data is in the `TFRecord` format, we need to download a pre-trained model checkpoint and then configure the training pipeline.

In [ ]:
# Download a pre-trained model checkpoint

# Choose a model from the TensorFlow 2 Detection Model Zoo:
# https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md
# For MobileNetV2, a good option is 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'

model_name = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
base_url = 'http://download.tensorflow.org/models/object_detection/tf2/20200711/'
model_archive_name = f'{model_name}.tar.gz'

# Create a directory to store the pre-trained model
pretrained_model_dir = '/content/pretrained_model'
os.makedirs(pretrained_model_dir, exist_ok=True)

# Download the model archive
!wget -q {base_url}{model_archive_name} -O {pretrained_model_dir}/{model_archive_name}

# Extract the model archive
!tar -xf {pretrained_model_dir}/{model_archive_name} -C {pretrained_model_dir}

print(f"Pre-trained model '{model_name}' downloaded and extracted to {pretrained_model_dir}/")

# Define the checkpoint path
pretrained_checkpoint_path = os.path.join(pretrained_model_dir, model_name, 'checkpoint', 'ckpt-0')
print(f"Pre-trained checkpoint path: {pretrained_checkpoint_path}")

Finally, we need to create a pipeline configuration file. This file tells TFODAPI how to train your model, including paths to your data, label map, pre-trained model, and training parameters. We will start with a template config file and modify it.

In [ ]:
import tensorflow as tf
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2
import os

# Paths (using existing variables)
# output_dir = '/content/data/tfodapi_data'
# label_map_path = os.path.join(output_dir, 'label_map.pbtxt')
# train_record_path = os.path.join(output_dir, 'train.record')
# val_record_path = os.path.join(output_dir, 'val.record')
# pretrained_checkpoint_path = os.path.join(pretrained_model_dir, model_name, 'checkpoint', 'ckpt-0')

# Path to the base config file for the chosen model
config_template_path = os.path.join(pretrained_model_dir, model_name, 'pipeline.config')

# Output path for the new custom config file
custom_config_path = os.path.join(output_dir, 'custom_pipeline.config')

# Load the base config file
config = pipeline_pb2.TrainEvalPipelineConfig()
with tf.io.gfile.GFile(config_template_path, "r") as f:
    text_format.Merge(f.read(), config)

# Modify the config file
# Number of classes
config.model.ssd.num_classes = len(label_map)

# Fine-tune checkpoint
config.train_config.fine_tune_checkpoint = pretrained_checkpoint_path
config.train_config.fine_tune_checkpoint_type = 'detection'
config.train_config.batch_size = 1 # Keep batch size small to avoid OOM

# Training data input reader
config.train_input_reader.label_map_path = label_map_path
config.train_input_reader.tf_record_input_reader.input_path[:] = [train_record_path]

# Validation data input reader
config.eval_input_reader[0].label_map_path = label_map_path
config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [val_record_path]

# Set training steps and evaluation interval (adjust as needed)
config.train_config.num_steps = 20000 # Example: 20,000 steps
config.eval_config.num_examples = 500 # Evaluate every 500 steps

# Write the modified config to a new file
with tf.io.gfile.GFile(custom_config_path, "w") as f:
    f.write(text_format.MessageToString(config))

print(f"Custom pipeline configuration saved to: {custom_config_path}")

# Set up the model directory for training outputs
model_dir = '/content/training_output'
os.makedirs(model_dir, exist_ok=True)
print(f"Model training output will be saved to: {model_dir}")

#### STEP 5 — Train MobileNetV2 Model (for Object Detection)

Now we'll use `tensorflow-lite-model-maker` to train an `SSD MobileNetV2` or `EfficientDet Lite` model for object detection. We'll use the `object_detector` module.

In [ ]:
# Explicitly upgrade tensorflow_estimator to match the installed TensorFlow version (2.20.0)
!/content/.venv/bin/pip install --upgrade tensorflow_estimator==2.20.0

print("tensorflow_estimator upgraded to 2.20.0. Please re-run the TFODAPI verification cell (`f3394e7e`) and then the training cell (`3ea6e3e4`).")

In [ ]:
# Initiate training using the TFODAPI's model_main_tf2.py script
import os

# Navigate to the models/research directory where model_main_tf2.py is located
%cd /content/models/research

# Install missing 'lvis' dependency for TFODAPI (already satisfied, but keep for robustness)
!/content/.venv/bin/pip install lvis

# Install tensorflow_estimator to provide 'estimator' module for TF2 compatibility (already satisfied, but ensure)
!/content/.venv/bin/pip install tensorflow_estimator

# Install/downgrade other dependencies required by object_detection 0.1
# Only install apache-beam, avro-python3, contextlib2. Remove version pinning for pyparsing and sacrebleu
# as they cause conflicts with matplotlib and tf-models-official.
print("Installing/downgrading additional object_detection dependencies...")
!/content/.venv/bin/pip install \
    apache-beam \
    avro-python3 \
    contextlib2 \
    --upgrade

# Define paths (these are already defined in the notebook's global state)
# custom_config_path = '/content/data/tfodapi_data/custom_pipeline.config'
# model_dir = '/content/training_output'

# Run the training script
# The TFODAPI training script (model_main_tf2.py) expects a path to the pipeline config file
# and a directory where it can save checkpoints and logs.
print(f"Starting TFODAPI training with config: {custom_config_path} and model_dir: {model_dir}")

# Use the virtual environment's python interpreter to run the training script
# Set MPLBACKEND to 'agg' to avoid backend issues with matplotlib in non-interactive environments
!MPLBACKEND=agg /content/.venv/bin/python object_detection/model_main_tf2.py \
    --model_dir={model_dir} \
    --pipeline_config_path={custom_config_path} \
    --num_train_steps={config.train_config.num_steps} \
    --checkpoint_every_n=1000 \
    --record_summaries=True